# 01 · Inspect the vulnerability dataset

A reproducible first pass over the bundled CVE snapshot: inspect its schema, missing values, score coverage, publication years, and common CWE labels. This notebook does not change the raw CSV or make severity/modeling assumptions.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Find the repository root whether Jupyter starts here or in notebooks/.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "raw" / "vulnerabilities.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate data/raw/vulnerabilities.csv")

sys.path.insert(0, str(PROJECT_ROOT))
from src import load_vulnerabilities, summarize_missingness

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "vulnerabilities.csv"
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
df = load_vulnerabilities(DATA_PATH)
print(f"Records: {len(df):,} | Fields: {len(df.columns)}")
print("Columns:", ", ".join(df.columns))
df.head()

## Schema and data quality

The raw identifier column has a blank header; the loader exposes it as `cve_id`. Dates and CVSS scores are parsed for inspection, while the on-disk CSV remains untouched.

In [ ]:
df.info()
summarize_missingness(df).query("missing_count > 0")

In [ ]:
# Numeric summary for CVSS; inspect date coverage separately.
df["cvss"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df["cvss"].dropna().plot.hist(bins=20, ax=ax, color="#4263eb", edgecolor="white")
ax.set(title="CVSS score distribution", xlabel="CVSS score as supplied", ylabel="Record count")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()

## Publication timeline and CWE labels

These plots describe the snapshot as supplied. They should not be interpreted as real-world incident rates or current vulnerability activity.

In [ ]:
publication_years = df["pub_date"].dropna().dt.year.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 4))
publication_years.plot(ax=ax, color="#12b886")
ax.set(title="Records by publication year", xlabel="Publication year", ylabel="Record count")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()

In [ ]:
cwe_counts = (
    df.groupby(["cwe_code", "cwe_name"], dropna=False)
      .size()
      .sort_values(ascending=False)
      .head(15)
      .rename("record_count")
      .reset_index()
)
cwe_counts

## Initial notes

- The bundled data is a historical snapshot; its publication dates end in 2019.
- Some access/impact attributes are missing. Treat blanks as unknown, not as zero or as an absence of risk.
- The file does not specify the CVSS version. Do not apply version-specific severity cutoffs until provenance is established.
- Before downstream modeling, check duplicates, temporal coverage, label quality, and any survivorship or reporting bias relevant to the question.